# 11 — SQL on my transactions

Same questions as my pandas work, answered in SQL. SQLite is built into Python: no server, the database lives in memory.

Covers: `GROUP BY`, `HAVING`, CTEs (`WITH`), window functions (`ROW_NUMBER`, `LAG`, running `SUM`), and `JOIN`. Each answer is cross-checked against pandas.

In [1]:
import sqlite3
import sys
import pandas as pd

sys.path.insert(0, '../service')
from clean import clean_description

conn = sqlite3.connect(':memory:')
# Python function usable inside SQL: same cleaning the model uses (strips *1234 and branch names)
conn.create_function('clean', 1, lambda s: clean_description(s or ''))

tx = pd.read_csv('../data/fake_transactions_3y.csv')  # Date stays an ISO string, e.g. 2026-10-20T19:42:00.000Z
tx.to_sql('transactions', conn, index=False)

def sql(query):
    return pd.read_sql_query(query, conn)

print('SQLite', sqlite3.sqlite_version)
sql('SELECT COUNT(*) AS rows, MIN(Date) AS first, MAX(Date) AS last FROM transactions')

SQLite 3.50.4


,rows,first,last
0,4410,2023-10-22T09:37:00.000Z,2026-10-20T19:42:00.000Z


## 1. Spending by category — `GROUP BY`

`WHERE` filters rows *before* grouping; aggregates (`SUM`, `COUNT`, `AVG`) collapse each group to one row.

In [2]:
by_cat = sql('''
    SELECT Category,
           COUNT(*)              AS n,
           ROUND(SUM(Amount), 2) AS total,
           ROUND(AVG(Amount), 2) AS avg
    FROM transactions
    WHERE Type = 'expense'
    GROUP BY Category
    ORDER BY total DESC
''')

# cross-check with pandas
expected = tx[tx.Type == 'expense'].groupby('Category')['Amount'].sum().round(2)
assert by_cat.set_index('Category')['total'].sort_index().equals(expected.sort_index())
by_cat

,Category,n,total,avg
0,Groceries,365,723931.26,1983.37
1,Shopping,272,641795.05,2359.54
2,Transportation,1095,573438.57,523.69
3,Food & Drinks,1415,536734.70,379.32
4,Bills & Utilities,216,257145.27,1190.49
5,Entertainment,125,149843.54,1198.75
6,Healthcare,91,122799.29,1349.44
7,Coffee & Snacks,697,110279.50,158.22


## 2. Monthly totals + month-over-month change — CTE and `LAG`

A **CTE** (`WITH monthly AS (...)`) names a sub-result so the next query can read it like a table.
`LAG(x) OVER (ORDER BY month)` reads the *previous row's* value without collapsing rows — that's what makes it a **window function**.

In [3]:
monthly = sql('''
    WITH monthly AS (
        SELECT substr(Date, 1, 7) AS month,
               ROUND(SUM(Amount), 2) AS spent
        FROM transactions
        WHERE Type = 'expense'
        GROUP BY month
    )
    SELECT month,
           spent,
           LAG(spent) OVER (ORDER BY month) AS prev,
           ROUND(100.0 * (spent - LAG(spent) OVER (ORDER BY month))
                 / LAG(spent) OVER (ORDER BY month), 1) AS pct_change
    FROM monthly
    ORDER BY month
''')

expected = tx[tx.Type == 'expense'].groupby(tx.Date.str[:7])['Amount'].sum().round(2)
assert monthly.set_index('month')['spent'].equals(expected.rename_axis('month').rename('spent'))
monthly.tail(12)

,month,spent,prev,pct_change
25,2025-11,98054.01,87653.17,11.9
26,2025-12,107246.79,98054.01,9.4
27,2026-01,85219.64,107246.79,-20.5
28,2026-02,67777.53,85219.64,-20.5
29,2026-03,108494.18,67777.53,60.1
30,2026-04,86733.84,108494.18,-20.1
31,2026-05,84602.00,86733.84,-2.5
32,2026-06,88253.34,84602.00,4.3
33,2026-07,91994.48,88253.34,4.2
34,2026-08,73728.83,91994.48,-19.9


## 3. Top 3 merchants per category — `ROW_NUMBER() OVER (PARTITION BY ...)`

`PARTITION BY Category` restarts the numbering for each category. Classic interview question: "top N per group".
`clean(Description)` is my Python function, so `PETRON*2219` and `Petron` count as one merchant.

In [4]:
top_merchants = sql('''
    WITH merchant_totals AS (
        SELECT Category,
               clean(Description) AS merchant,
               COUNT(*)              AS visits,
               ROUND(SUM(Amount), 2) AS total
        FROM transactions
        WHERE Type = 'expense' AND clean(Description) <> ''
        GROUP BY Category, merchant
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY Category ORDER BY total DESC) AS rank
        FROM merchant_totals
    )
    SELECT Category, rank, merchant, visits, total
    FROM ranked
    WHERE rank <= 3
    ORDER BY Category, rank
''')
assert top_merchants.groupby('Category')['rank'].max().le(3).all()
top_merchants

,Category,rank,merchant,visits,total
0,Bills & Utilities,1,meralco,36,104474.37
1,Bills & Utilities,2,pldt home fiber,29,53047.87
2,Bills & Utilities,3,globe postpaid,33,35387.27
3,Coffee & Snacks,1,dunkin,83,14206.85
4,Coffee & Snacks,2,-eleven,85,12589.24
5,Coffee & Snacks,3,potato corner,80,12495.96
6,Entertainment,1,concert ticket,27,31266.77
7,Entertainment,2,steam,21,26042.98
8,Entertainment,3,videoke,21,23393.36
9,Food & Drinks,1,carinderia lunch,116,49240.33


## 4. Running total this year — `SUM() OVER (ORDER BY ...)`

Same `SUM`, but as a window: each row adds itself to everything before it.

In [5]:
running = sql('''
    SELECT substr(Date, 1, 7) AS month,
           ROUND(SUM(Amount), 2) AS spent,
           ROUND(SUM(SUM(Amount)) OVER (ORDER BY substr(Date, 1, 7)), 2) AS year_to_date
    FROM transactions
    WHERE Type = 'expense' AND Date >= '2026-01-01'
    GROUP BY month
    ORDER BY month
''')
assert abs(running['year_to_date'].iloc[-1] - running['spent'].sum()) < 0.05
running

,month,spent,year_to_date
0,2026-01,85219.64,85219.64
1,2026-02,67777.53,152997.17
2,2026-03,108494.18,261491.35
3,2026-04,86733.84,348225.19
4,2026-05,84602.00,432827.19
5,2026-06,88253.34,521080.53
6,2026-07,91994.48,613075.01
7,2026-08,73728.83,686803.84
8,2026-09,102778.20,789582.04
9,2026-10,65003.33,854585.37


## 5. Over budget — `JOIN` + `HAVING`

A second table: my monthly budget per category. `JOIN` matches rows on `Category`.
`HAVING` filters *after* grouping (you can't use `WHERE` on `SUM(...)`).

`LEFT JOIN` vs `JOIN`: a `LEFT JOIN` keeps categories with no budget row (budget = NULL); an inner `JOIN` drops them.

In [6]:
budgets = pd.DataFrame({
    'Category': ['Food & Drinks', 'Coffee & Snacks', 'Groceries', 'Transportation',
                 'Bills & Utilities', 'Shopping', 'Healthcare'],   # no budget for Entertainment on purpose
    'monthly_limit': [16000, 3300, 22000, 17000, 8000, 19000, 4000],  # ~10% above my average month
})
budgets.to_sql('budgets', conn, index=False)

over = sql('''
    SELECT substr(t.Date, 1, 7)      AS month,
           t.Category,
           ROUND(SUM(t.Amount), 2)   AS spent,
           b.monthly_limit,
           ROUND(SUM(t.Amount) - b.monthly_limit, 2) AS over_by
    FROM transactions t
    JOIN budgets b ON b.Category = t.Category
    WHERE t.Type = 'expense'
    GROUP BY month, t.Category
    HAVING SUM(t.Amount) > b.monthly_limit
    ORDER BY over_by DESC
''')
print(f'{len(over)} category-months over budget')
over.head(10)

72 category-months over budget


,month,Category,spent,monthly_limit,over_by
0,2026-03,Groceries,41697.31,22000,19697.31
1,2024-01,Shopping,38367.35,19000,19367.35
2,2026-09,Shopping,37343.72,19000,18343.72
3,2024-03,Shopping,33609.00,19000,14609.00
4,2024-05,Groceries,35134.54,22000,13134.54
5,2024-04,Shopping,29156.37,19000,10156.37
6,2023-12,Shopping,27753.85,19000,8753.85
7,2025-05,Groceries,30723.25,22000,8723.25
8,2025-12,Shopping,27082.67,19000,8082.67
9,2025-10,Groceries,29813.99,22000,7813.99


In [7]:
# How often is each category over budget? LEFT JOIN so unbudgeted categories still show up.
sql('''
    WITH cat_month AS (
        SELECT substr(Date, 1, 7) AS month, Category, SUM(Amount) AS spent
        FROM transactions
        WHERE Type = 'expense'
        GROUP BY month, Category
    )
    SELECT c.Category,
           b.monthly_limit,
           COUNT(*)                                                  AS months,
           SUM(CASE WHEN c.spent > b.monthly_limit THEN 1 ELSE 0 END) AS months_over,
           ROUND(AVG(c.spent), 2)                                    AS avg_spent
    FROM cat_month c
    LEFT JOIN budgets b ON b.Category = c.Category
    GROUP BY c.Category
    ORDER BY months_over DESC
''')

,Category,monthly_limit,months,months_over,avg_spent
0,Shopping,19000.0,37,13,17345.81
1,Groceries,22000.0,37,13,19565.71
2,Healthcare,4000.0,35,12,3508.55
3,Food & Drinks,16000.0,37,12,14506.34
4,Coffee & Snacks,3300.0,37,10,2980.53
5,Transportation,17000.0,37,7,15498.34
6,Bills & Utilities,8000.0,36,5,7142.92
7,Entertainment,NaN,35,0,4281.24


## What I learned

TODO (me): 3-5 lines.
- When is SQL nicer than pandas here, and when is pandas nicer?
- `WHERE` vs `HAVING` in my own words.
- What does `PARTITION BY` do that `GROUP BY` can't?